# Lesson 2.8 — 草稿 notebook：环境、trajectory 与 expert planner

这是一个**工作用 notebook**，而不是整理过的课程。它保留了项目找出如何产出一个成功的 PickCube demonstration 的完整路径，包括走过的弯路。这里的一切都是分三轮写成的：

1. **环境基础**（2.8.1–2.8.2）—— 构建 `PickCube-v1`，读取 42 维 observation，采样并 step 一个 action。
2. **随机 trajectory 采集** —— 组装一个 transition 列表和一个完整 episode。
3. **寻找 planner** —— 在已安装的 ManiSkill 包中搜索 demonstration 生成器，转向 `pd_joint_pos`，并构建 planner。

> **关于顺序的说明。** 原文件末尾的三处 import 自检（`mplib`、`sapien`、planner 类）已被**上移**，放到它们所支撑的 planner 构建之前。它们的代码未变，记录的 outputs 也是原始结果，因此 `execution_count` 的取值现在看起来是乱序的。除此之外，cell 的 source 与 outputs 未作其他修改。

## 为什么 `pd_joint_pos` 会出现在中间

第一轮使用的是 `pd_joint_delta_pos`，也就是项目 dataset fixture 的控制模式。planner 后来拒绝了它：它的 gripper 辅助函数输出 `[qpos(7), gripper]`，即一个 8 维的 **absolute** 位置 action，因此 delta 模式的环境会失败并报 `Received action of shape torch.Size([15]) but expected shape (1, 8)`。后半部分切换到 `pd_joint_pos`，正是对这一发现过程的记录。

> **环境要求。** 构建 planner 需要 `mplib`，而它是针对 NumPy 1.x 的 C API 编译的。在 NumPy 2.x 的环境中，本 notebook 的 planner cell 会以 `SIGSEGV`（地址 `0x0`）终止 kernel。请在 `embodied310`（NumPy 1.26.4）下运行 planner 相关的 cells；参见 `notes/progress.md`。

成功路径的整理版本位于 `2.9_expert_demonstrations.ipynb`；批量采集器是 `scripts/generate_expert_demo.py`。

## 2.8.1 — 构建环境

三个参数决定了 agent 看到什么、做什么：

- `obs_mode="state"` —— 展平的 robot + object state vector，不含图像。
- `control_mode="pd_joint_delta_pos"` —— action 是**关节位置的变化量**，也就是项目 dataset fixture 使用的模式。
- `render_mode="rgb_array"` —— 在 notebook 中无需 display server 即可工作。

In [2]:
import gymnasium as gym

import mani_skill.envs


# --------------------------------------------------
# Create PickCube environment
#
# obs_mode:
#   state = robot + object information
#
# control_mode:
#   pd_joint_delta_pos
#   means action controls joint position change
#
# render_mode:
#   rgb_array works better in notebook
# --------------------------------------------------

env = gym.make(
    "PickCube-v1",
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
    render_mode="rgb_array",
)


print("Environment initialized")

print("Observation:")
print(env.observation_space)

print("Action:")
print(env.action_space)

Environment initialized
Observation:
Box(-inf, inf, (1, 42), float32)
Action:
Box(-1.0, 1.0, (8,), float32)


In [3]:
# --------------------------------------------------
# Reset simulator
#
# This creates:
# - physics scene
# - robot
# - objects
# - task state
# --------------------------------------------------

obs, info = env.reset()


print("Reset successful!")

print("Observation shape:")
print(obs.shape)

Reset successful!
Observation shape:
torch.Size([1, 42])


### Reset 并读取 42 维 observation

`env.reset()` 构建物理场景、robot、objects 以及 task state。返回的 tensor 形状为 `(num_envs, 42)`：首轴之所以存在，是因为 ManiSkill 是向量化的，而这里 `num_envs=1`。

`obs_mode="state"` 只返回一个扁平 vector，因此这 42 个数字背后的结构必须手工逐层剥开。已验证的布局为

```text
[0:9]   qpos
[9:18]  qvel
[18:19] is_grasped
[19:26] tcp_pose
[26:29] goal_pos
[29:36] obj_pose
[36:39] tcp_to_obj_pos
[39:42] obj_to_goal_pos
```

In [4]:
# Print the raw observation tensor
#
# obs shape:
# (num_envs, state_dimension)
#
# Here:
# num_envs = 1
# state_dimension = 42


print(obs)

print("-------------------")

print("dtype:")
print(obs.dtype)

print("-------------------")

print("device:")
print(obs.device)

tensor([[ 0.0076,  0.3909, -0.0461, -1.9406, -0.0307,  2.3389,  0.8057,  0.0400,
          0.0400,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0058, -0.0273,  0.1793,  0.0199,  0.9997,
         -0.0170,  0.0036,  0.0530,  0.0357,  0.2286, -0.0760, -0.0574,  0.0200,
          0.1607, -0.0000, -0.0000, -0.9870, -0.0818, -0.0301, -0.1593,  0.1290,
          0.0931,  0.2086]])
-------------------
dtype:
torch.float32
-------------------
device:
cpu


In [5]:
# Convert torch tensor to numpy array
#
# detach:
# remove gradient tracking
#
# cpu:
# move data from GPU to CPU
#
# numpy:
# convert to numpy format


state = (
    obs
    .detach()
    .cpu()
    .numpy()
)


print(state.shape)

(1, 42)


In [6]:
def analyze_pickcube_state(state):
    """
    Analyze PickCube-v1 state observation.

    Current observation contains:
    - robot joint information
    - end-effector pose
    - object pose
    - goal information
    - gripper state

    This helps us understand:
    what information a robot policy receives.
    """

    # Remove environment dimension
    # shape:
    # (1,42) -> (42,)

    state = state[0]


    print("Total dimensions:")
    print(len(state))


    print("\n========== Robot Joint Position ==========")

    # First 9 dimensions:
    # robot joint position information

    print(state[:9])


    print("\n========== Robot Joint Velocity ==========")

    # Next 9 dimensions:
    # robot joint velocity

    print(state[9:18])


    print("\n========== End Effector Pose ==========")

    # Position + orientation

    print(state[18:25])


    print("\n========== Object Pose ==========")

    # Cube position and orientation

    print(state[25:32])


    print("\n========== Goal ==========")

    # Target position

    print(state[32:35])


    print("\n========== Relative Information ==========")

    # Relative distance information

    print(state[35:41])


    print("\n========== Gripper ==========")

    # Open / close state

    print(state[41])

In [7]:
analyze_pickcube_state(state)

Total dimensions:
42

========== Robot Joint Position ==========
[ 0.00755627  0.39089483 -0.04611887 -1.9406402  -0.03071309  2.3389194
  0.8057291   0.04        0.04      ]

========== Robot Joint Velocity ==========
[0. 0. 0. 0. 0. 0. 0. 0. 0.]

========== End Effector Pose ==========
[ 0.          0.0057635  -0.02726336  0.17932712  0.01989457  0.9996503
 -0.01704317]

========== Object Pose ==========
[ 0.00363891  0.05301899  0.03570371  0.22857928 -0.07601617 -0.05735836
  0.02      ]

========== Goal ==========
[ 0.1606978 -0.        -0.       ]

========== Relative Information ==========
[-0.9870037  -0.08177967 -0.030095   -0.15932712  0.12903516  0.09306207]

========== Gripper ==========
0.20857929


## 2.8.2 — Action 与 step 循环

`env.action_space.sample()` 从 `[-1, 1]^8` 中均匀随机采样一个 action。这是一个 **pipeline fixture**，而不是 policy：由此得到的 trajectory 具备合法的结构，但基本上没有有用的行为。

`env.step(action)` 返回五个值。其中两个是彼此独立的停止条件，很容易混为一谈：

- `terminated` —— 任务以成功或不可恢复的失败结束；
- `truncated` —— episode 触及了 step 数或时间上限，这与是否成功无关。

In [8]:
# Sample one random action

action = env.action_space.sample()

print("Action:")
print(action)

print("----------------")

print("Shape:")
print(action.shape)

Action:
[ 0.31634188 -0.9451699   0.25665474 -0.64098585 -0.9949915  -0.92318964
  0.04182364 -0.71126246]
----------------
Shape:
(8,)


### 关于重复的采样 cell

上面的 cell 和下面的 cell 都调用了 `env.action_space.sample()`。前者是最小版本；后者重复了一次，并把 shape 检查明确写了出来。这一重复被作为原始记录保留 —— 两者内容完全相同。

In [9]:
# --------------------------------------------------
# Sample one action from ManiSkill action space
#
# Action represents:
# "what the robot should do at this timestep"
#
# Current control mode:
# pd_joint_delta_pos
#
# Therefore:
# action = desired joint position change
# --------------------------------------------------


action = env.action_space.sample()


print("Action:")
print(action)


print("------------------------")


print("Action shape:")
print(action.shape)

Action:
[-0.97418123  0.1994947   0.4745303  -0.02078718  0.26149392 -0.4823804
 -0.17995225 -0.9917719 ]
------------------------
Action shape:
(8,)


两个 cell 都从同一个 space 中采样一个 action，因此这里打印出的 shape 就是 action 的约定：位于 `[-1, 1]` 中的 `(8,)`，由向量化的 `Box(-1.0, 1.0, (8,), float32)` reshape 为 `(8,)`。

In [10]:
# --------------------------------------------------
# Execute one robot action
#
# env.step(action):
#
# input:
#     action_t
#
# output:
#     next observation
#     reward
#     termination signal
# --------------------------------------------------


next_obs, reward, terminated, truncated, info = env.step(action)


print("Next observation:")
print(next_obs.shape)


print("----------------")


print("Reward:")
print(reward)


print("----------------")


print("Info:")
print(info)

Next observation:
torch.Size([1, 42])
----------------
Reward:
tensor([0.0591])
----------------
Info:
{'elapsed_steps': tensor([1], dtype=torch.int32), 'success': tensor([False]), 'is_obj_placed': tensor([False]), 'is_robot_static': tensor([False]), 'is_grasped': tensor([False])}


## 2.8.3 — 采集一个随机 episode

下面两个代码块构建的是同一件事，只是细致程度不同：先是仅 20 步的 `(observation, action, reward)` 裸列表，然后是一个完整 episode，它还会记录 success 标志并在 `terminated` / `truncated` 时停止。

这个 list-of-dicts **就是**最小的 robot dataset 格式。下游的一切（HDF5、LeRobot、DataLoader）都只是围绕它的簿记工作。

存储的 observation 是施加 action **之前**的 state，这正是 `(o_t, a_t)` 配对约定的由来。

### 为什么同时检查 `terminated` 和 `truncated`

只依据 `terminated` 停止会越过时间上限；只依据 `truncated` 停止则会忽略提前到来的成功。循环在两者任一满足时退出，这才使记录到的长度有意义。

注意这个循环确立的配对约定：在第 `t` 步追加的 observation 是该步**进入时**的 state，因此存储的列表是 `(o_t, a_t)` —— 而不是 `(o_t, a_{t+1})`。

### 最小 transition 列表

最小可用的容器：三个并列的列表。它只记录循环所消耗和产生的全部内容，别无其他 —— 没有 success 标志，也没有停止条件。

In [11]:
# --------------------------------------------------
# Collect one short trajectory
#
# We store:
# observation
# action
# reward
#
# This is the minimal robot dataset format
# --------------------------------------------------


trajectory = {
    "observations": [],
    "actions": [],
    "rewards": []
}


# Reset environment

obs, info = env.reset()


for step in range(20):

    # Random action for now
    # Later replaced by expert action

    action = env.action_space.sample()


    next_obs, reward, terminated, truncated, info = env.step(action)


    # Store current transition

    trajectory["observations"].append(
        obs.cpu().numpy()
    )


    trajectory["actions"].append(
        action
    )


    trajectory["rewards"].append(
        reward
    )


    obs = next_obs


    if terminated or truncated:
        break



print(
    "Collected steps:",
    len(trajectory["actions"])
)

Collected steps: 20


### 如实解读结果

这里得到的是一个**随机** episode：总 reward 很小，本 notebook 中没有任何 episode 成功。这是预期之内的，也正是重点所在 —— 这条 trajectory 用来验证 pipeline，它无法训练 policy。

区分随机数据与 expert 数据的信号是 action 的平滑度，而不是 reward：随机 action 在相邻 step 之间的跳变为 `mean |Δa| ≈ 0.67`，而 expert planner 的 action 为 `≈ 0.0078`。参见 `2.9_expert_demonstrations.ipynb`。

In [12]:
print(trajectory["rewards"])
print("Total reward:",
      sum(trajectory["rewards"]))

[tensor([0.0497]), tensor([0.0477]), tensor([0.0461]), tensor([0.0463]), tensor([0.0442]), tensor([0.0354]), tensor([0.0342]), tensor([0.0379]), tensor([0.0408]), tensor([0.0395]), tensor([0.0356]), tensor([0.0325]), tensor([0.0349]), tensor([0.0389]), tensor([0.0411]), tensor([0.0501]), tensor([0.0480]), tensor([0.0379]), tensor([0.0353]), tensor([0.0341])]
Total reward: tensor([0.8100])


### 带停止条件的完整 episode

思路相同，但它还会存储 `success`，并在 `terminated` / `truncated` 时停止，因此返回的长度反映的是真实的 episode 边界，而不是固定的 step 预算。

In [13]:
# --------------------------------------------------
# Collect one complete episode
#
# We record:
# - observation
# - action
# - reward
# - success
#
# This format is closer to real robot datasets
# --------------------------------------------------


def collect_random_episode(env, max_steps=200):

    trajectory = {
        "observations": [],
        "actions": [],
        "rewards": [],
        "success": False,
    }


    obs, info = env.reset()


    for step in range(max_steps):

        # Random policy
        action = env.action_space.sample()


        next_obs, reward, terminated, truncated, info = env.step(action)


        # Save transition

        trajectory["observations"].append(
            obs.cpu().numpy()
        )

        trajectory["actions"].append(
            action
        )

        trajectory["rewards"].append(
            reward.cpu().numpy()
        )


        obs = next_obs


        if terminated or truncated:

            trajectory["success"] = (
                info["success"].item()
            )

            break


    return trajectory

In [14]:
random_traj = collect_random_episode(env)

print(
    "Length:",
    len(random_traj["actions"])
)

print(
    "Success:",
    random_traj["success"]
)

Length: 50
Success: False


## 2.8.4 — 寻找 demonstration 生成器

ManiSkill 3 改变了它的 API，因此定位 planner 的方式是检查**已安装的包**，而不是相信较旧的教程。下面的搜索遍历 `mani_skill` 目录树，查找 demonstration 文件、agents、motion-planning 模块，最后是 `PickCube` 实现本身。

直接阅读 `motionplanner.py` 才是关键：它是生成 demonstration trajectory 的 expert controller，因此它的接口定义了成功的 episode 应当是什么样子。

In [15]:
import os
import mani_skill


# --------------------------------------------------
# Get ManiSkill installation directory
#
# We need this path to inspect:
# - demos
# - examples
# - utilities
# --------------------------------------------------

mani_skill_path = os.path.dirname(
    mani_skill.__file__
)


print("ManiSkill path:")
print(mani_skill_path)

ManiSkill path:
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill


上面的 cell 从导入的 module 计算出 `mani_skill_path`，因此这次搜索会跟随已安装包的实际所在位置。这是定位 ManiSkill 内部实现的通用做法 —— 不同于下面两格中硬编码的路径。

In [16]:
# --------------------------------------------------
# Search files related to demonstrations
#
# ManiSkill version 3 changed some APIs,
# so we inspect the installed package
# instead of assuming old tutorials.
# --------------------------------------------------

for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        if "demo" in file.lower():

            print(
                os.path.join(root, file)
            )

/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_vis_textures.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_vis_pcd.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_robot.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_random_action.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_reset_distribution.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_manual_control_continuous.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_vis_segmentation.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_manual_control.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/__pycach

In [17]:
# --------------------------------------------------
# Search robot agents
#
# ManiSkill uses agents to represent:
# - robot model
# - controller
# - action interface
# --------------------------------------------------

for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        if "agent" in file.lower():

            print(
                os.path.join(root, file)
            )

/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/base_agent.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/base_real_agent.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/multi_agent.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/__pycache__/multi_agent.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/__pycache__/base_agent.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/__pycache__/base_real_agent.cpython-312.pyc


In [18]:
# --------------------------------------------------
# Search motion planning related modules
#
# Expert demonstrations in simulation are usually
# generated by planners.
# --------------------------------------------------

keywords = [
    "motion",
    "planner",
    "planning"
]


for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        filename = file.lower()

        for keyword in keywords:

            if keyword in filename:

                print(
                    os.path.join(root, file)
                )

                break

/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/xarm6/motionplanner.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/xarm6/__pycache__/motionplanner.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/two_finger_gripper/motionplanner.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/two_finger_gripper/__pycache__/motionplanner.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/panda/motionplanner.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/panda/motionplanner_stick.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/panda/__pycache__/motionplanner_stick.cpython

In [19]:
# --------------------------------------------------
# Find PickCube implementation
# --------------------------------------------------

for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        if "pickcube" in file.lower():

            print(
                os.path.join(root, file)
            )

### 陷阱：这些路径是绝对的，且与环境相关

下面两处检查都通过一个**硬编码的绝对路径**打开文件，该路径嵌入了 `embodied` 环境的 Python 版本：

```text
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/...
```

这在本机上可行，但在任何其他 Python 版本、环境名或安装位置上都会失效。这也是下面那个 cell 中第二条路径带有第二个硬编码前缀的原因。更好的做法是从 module 推导路径，正如上面的搜索 cells 已经做的那样：

```python
path = Path(mani_skill.__file__).parent / "examples" / "motionplanning" / "..."
```

In [20]:
# --------------------------------------------------
# Inspect Panda motion planner
#
# This is the expert controller used for generating
# demonstration trajectories.
# --------------------------------------------------

planner_path = (
    "/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/"
    "mani_skill/examples/motionplanning/panda/motionplanner.py"
)


with open(planner_path, "r") as f:
    planner_code = f.read()


print(planner_code[:4000])

import mplib
import numpy as np
import sapien

from mani_skill.envs.sapien_env import BaseEnv
from mani_skill.examples.motionplanning.two_finger_gripper.motionplanner import TwoFingerGripperMotionPlanningSolver


class PandaArmMotionPlanningSolver(TwoFingerGripperMotionPlanningSolver):
    OPEN = 1
    CLOSED = -1
    MOVE_GROUP = "panda_hand_tcp"

    def __init__(
        self,
        env: BaseEnv,
        debug: bool = False,
        vis: bool = True,
        base_pose: sapien.Pose = None,  # TODO mplib doesn't support robot base being anywhere but 0
        visualize_target_grasp_pose: bool = True,
        print_env_info: bool = True,
        joint_vel_limits=0.9,
        joint_acc_limits=0.9,
    ):
        super().__init__(env, debug, vis, base_pose, visualize_target_grasp_pose, print_env_info, joint_vel_limits, joint_acc_limits)


In [21]:
planner_path = "/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/two_finger_gripper/motionplanner.py"

with open(planner_path, "r") as f:
    code = f.read()

print(code[:8000])

import mplib
import numpy as np
import sapien

from mani_skill.envs.sapien_env import BaseEnv
from mani_skill.envs.scene import ManiSkillScene
from mani_skill.examples.motionplanning.base_motionplanner.motionplanner import BaseMotionPlanningSolver
from transforms3d import quaternions


class TwoFingerGripperMotionPlanningSolver(BaseMotionPlanningSolver):
    OPEN = 1
    CLOSED = -1

    def __init__(
        self,
        env: BaseEnv,
        debug: bool = False,
        vis: bool = True,
        base_pose: sapien.Pose = None,  # TODO mplib doesn't support robot base being anywhere but 0
        visualize_target_grasp_pose: bool = True,
        print_env_info: bool = True,
        joint_vel_limits=0.9,
        joint_acc_limits=0.9,
    ):
        super().__init__(env, debug, vis, base_pose, print_env_info, joint_vel_limits, joint_acc_limits)
        self.gripper_state = self.OPEN
        self.visualize_target_grasp_pose = visualize_target_grasp_pose
        self.grasp_pose_visual = N

## 2.8.5 — Motion planning 需要 `pd_joint_pos`

第一轮正是在这里撞上了这一约束。

`env.unwrapped` 去掉 Gymnasium wrapper 以触及 ManiSkill API —— agent、robot、它的 links 与 joints，以及 planner 所需的 base pose。在 delta 模式的环境上，planner 的 action 约定无法被满足。

下面 `pd_joint_pos` 环境被创建了两次（一次用于检查，一次用于 planning 运行）。这一重复被保留：它是原始可用的记录，而第二个代码块才是 planner 所绑定的那一个。

In [2]:
# ==========================================
# Step 0:
# Create ManiSkill environment
# ==========================================

import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
    render_mode="rgb_array",
)


print("Environment created")
print(type(env))

Environment created
<class 'mani_skill.utils.registration.TimeLimitWrapper'>


In [3]:
# ==========================================
# Step 1:
# Remove Gymnasium wrapper
# ==========================================


real_env = env.unwrapped


print("Environment:")
print(type(real_env))


print("----------------")


print("Robot agent:")
print(type(real_env.agent))

Environment:
<class 'mani_skill.envs.tasks.tabletop.pick_cube.PickCubeEnv'>
----------------
Robot agent:
<class 'mani_skill.agents.robots.panda.panda.Panda'>


In [5]:
# ==========================================
# Step 2:
# Inspect Panda robot
# ==========================================


robot = real_env.agent.robot


print("Robot:")
print(robot)


print("----------------")


print("Robot attributes:")
print(
    [x for x in dir(robot) if "pose" in x.lower()]
)

Robot:
<panda: struct of type <class 'mani_skill.utils.structs.articulation.Articulation'>; managing 1 <class 'sapien.pysapien.physx.PhysxArticulation'> objects>
----------------
Robot attributes:
['get_pose', 'get_root_pose', 'initial_pose', 'pose', 'root_pose', 'set_pose', 'set_root_pose']


In [6]:
# ==========================================
# Step 3:
# Get robot base pose
# ==========================================


robot_base_pose = robot.pose


print(robot_base_pose)

Pose(raw_pose=tensor([[-6.1500e-01,  7.2760e-11, -1.4901e-08,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00]]))


In [7]:
print(real_env.control_mode)

pd_joint_delta_pos


下面的注释 —— *"Motion planner outputs joint positions"* —— 正是这第二个环境存在的全部原因。也就是 2.8.1 中 `pd_joint_delta_pos` 的假设被打破的那一刻。

In [8]:
import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",
    obs_mode="state",

    # Important:
    # Motion planner outputs joint positions.
    # This cell only inspects the mode; the planner is bound to the env created below.
    control_mode="pd_joint_pos",

    render_mode="rgb_array",
)


print(env)

<TimeLimitWrapper<OrderEnforcing<PickCubeEnv<PickCube-v1>>>>


In [9]:
real_env = env.unwrapped

print(real_env.control_mode)

pd_joint_pos


### 两个 `pd_joint_pos` 环境各自的作用

- 上面紧邻的 cell 用于**检查**模式（`print(real_env.control_mode)`）；
- 下面的 cell 才是 planner 实际**绑定**的那一个，其中的 `real_env` 被用于 `reset`、`robot_base_pose` 以及 planner 构造函数。

两者设置的选项完全相同，这一重复被作为原始记录保留。重新运行 notebook 时，请注意后面的 cells 闭包捕获的是哪一个 `env`。

In [11]:
# ==================================================
# Create PickCube environment for Motion Planning
#
# Important:
# Motion Planner outputs joint positions.
#
# Therefore:
# control_mode = pd_joint_pos
#
# ==================================================

import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",

    # We first use state observation
    # because motion planner does not need images
    obs_mode="state",

    # Important:
    # Planner outputs joint position targets.
    # This is the environment the planner is bound to.
    control_mode="pd_joint_pos",

    render_mode="rgb_array",
)


print("Environment created")

print(type(env))

print("----------------")

print("Control mode:")
print(env.unwrapped.control_mode)

Environment created
<class 'mani_skill.utils.registration.TimeLimitWrapper'>
----------------
Control mode:
pd_joint_pos


In [12]:
# ==================================================
# Remove Gym wrapper
# ==================================================

real_env = env.unwrapped


print(type(real_env))

print(real_env.agent)

<class 'mani_skill.envs.tasks.tabletop.pick_cube.PickCubeEnv'>


In [13]:
# ==================================================
# Reset environment
#
# The planner needs:
# - current robot state
# - cube position
# - goal position
# ==================================================

obs, info = real_env.reset()


print("Observation:")
print(obs.shape)

print("----------------")

print(info)

Observation:
torch.Size([1, 42])
----------------
{'elapsed_steps': tensor([0], dtype=torch.int32), 'success': tensor([False]), 'is_obj_placed': tensor([False]), 'is_robot_static': tensor([True]), 'is_grasped': tensor([False]), 'reconfigure': False}


In [14]:
robot_base_pose = real_env.agent.robot.pose

print(robot_base_pose)

Pose(raw_pose=tensor([[-6.1500e-01,  7.2760e-11, -1.4901e-08,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00]]))


## 2.8.6 — 构建 expert planner

在构造任何东西之前，先确认各组件能够 import。这三个 cell 原本位于原文件末尾；它们被移到这里，因为它们是前置条件，而不是结论。

`PandaArmMotionPlanningSolver` 就是 expert controller。它针对**解包后**（unwrapped）的环境构造，并传入 robot base pose，因为除非另有说明，`mplib` 会假定 base 位于原点。

> **这就是那个需要 NumPy 1.x 的 cell。** 在 NumPy 2.x 下运行它，会让 kernel 在 `mplib` 内部发生 segmentation fault 而终止，且无法被 `try` / `except` 捕获。在 `embodied310`（NumPy 1.26.4）下它可以成功构建。

In [1]:
import mplib
print("mplib ok")

mplib ok


In [2]:
import sapien
print("sapien ok")

sapien ok


In [3]:
from mani_skill.examples.motionplanning.panda.motionplanner import PandaArmMotionPlanningSolver
print("planner class ok")

planner class ok


In [16]:
# ==========================================
# Import Panda Motion Planner
#
# This class provides the expert planner
# for Panda robot.
# ==========================================

from mani_skill.examples.motionplanning.panda.motionplanner import (
    PandaArmMotionPlanningSolver
)


print("PandaArmMotionPlanningSolver imported!")

PandaArmMotionPlanningSolver imported!


In [ ]:
import time


print("Start creating planner...")

start = time.time()


planner = PandaArmMotionPlanningSolver(
    real_env,

    # No extra debug information
    debug=False,

    # Disable visualization first
    vis=False,

    # Robot base coordinate
    base_pose=robot_base_pose
)


end = time.time()


print("----------------")
print("Planner created!")
print(
    "Initialization time:",
    end-start,
    "seconds"
)